In [ ]:
import torch
from torch.nn import functional as F

In [ ]:
with open("/content/drive/MyDrive/nutuk.txt") as f:
    text = f.read()


In [ ]:
print("length of the characters: ", len(text))

length of the characters:  1577732


In [ ]:
print(text[9000:10000])

sında
Diyarbekir (Vesika: 8, 9), Bitlis, Elaziz vilayetlerin­
de, İstanbul'dan idare olunan Kürt Teafi Cemiyeti!
vardı. Bu cemiyetin maksadı, yabancı himayesi altında bir Kürt hükümeti vü­
cuda getirmekti.
Konya ve havalisinde, İstanbul'dan idare olunan Teaiii İslam Cemiyeti teş­
kiline çalışılıyordu. Memleketin hemen her tarafında İtilaf ve Hürriyet, Sulh
ve Selamet cemiyetleri de vardı.

Memleket dahilinde
ve İstanbul'da milli
varlığa düşman
teşekküller

İngiliz Muhipleri
Cemiyeti

İstanbul'da, muhtelif maksatlarla gizli ve açık olmak
üzere de birtakım fırka veya cemiyet unvanı altında te­
şekküller vardı.
İstanbul'da mühim sayılacak teşebbüslerden biri İngiliz Muhipleri Cemiyeti
idi. Bu isimden, İngilizlere muhip2 olanların teşkil ettiği bir cemiyet anlaşılma­
sın! Bence, bu cemiyeti teşkil edenler, kendi şahıslannı ve şahsi menfaatlannı
sevenler ve şahıslanyla menfaatlannın dokunulmazlığı çaresini Loyd Core3 hü­
kümeti marifetiyle İngiliz himayesini teminde arayanlardır. Bu bedbaht

In [ ]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print("total chars: ", vocab_size)
print(''.join(chars))
print(vocab_size)

total chars:  103

 !"%'()*,-./0123456789:;<>?ABCDEFGHIJKLMNOPQRSTUVWYZ[\]_abcdefghijklmnopqrstuvwxyz{§«­·ÇÖÜçôöüğİıŞş•�
103


In [ ]:
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s] #encoder: take a string,output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) #decoder: take a list of integers, output a string

print(encode('noktali virgul'))
print(decode(encode('noktali virgul')))

[71, 72, 68, 77, 58, 69, 66, 2, 79, 66, 75, 64, 78, 69]
noktali virgul


In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)

data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[9000:10000])

torch.Size([1577732]) torch.int64
tensor([ 76,  98,  71,  61,  58,   0,  32,  66,  82,  58,  75,  59,  62,  68,
         66,  75,   2,   7,  50,  62,  76,  66,  68,  58,  24,   2,  22,  10,
          2,  23,   8,  10,   2,  30,  66,  77,  69,  66,  76,  10,   2,  33,
         69,  58,  83,  66,  83,   2,  79,  66,  69,  58,  82,  62,  77,  69,
         62,  75,  66,  71,  87,   0,  61,  62,  10,   2,  97,  76,  77,  58,
         71,  59,  78,  69,   6,  61,  58,  71,   2,  66,  61,  58,  75,  62,
          2,  72,  69,  78,  71,  58,  71,   2,  39,  95,  75,  77,   2,  48,
         62,  58,  63,  66,   2,  31,  62,  70,  66,  82,  62,  77,  66,   3,
          0,  79,  58,  75,  61,  98,  12,   2,  30,  78,   2,  60,  62,  70,
         66,  82,  62,  77,  66,  71,   2,  70,  58,  68,  76,  58,  61,  98,
         10,   2,  82,  58,  59,  58,  71,  60,  98,   2,  65,  66,  70,  58,
         82,  62,  76,  66,   2,  58,  69,  77,  98,  71,  61,  58,   2,  59,
         66,  75,   2,  39,  9

In [ ]:
#split
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [ ]:
block_size = 8 #we cannot give the entire data to model
train_data[:block_size+1], train_data[:block_size*2]

(tensor([42, 49, 48, 49, 39, 37,  0,  0,  7]),
 tensor([42, 49, 48, 49, 39, 37,  0,  0,  7, 15, 23, 15, 23, 11, 15, 23]))

In [ ]:
x = train_data[:block_size*2]
y = train_data[1:block_size*2+1]
for t in range(block_size):
  context = x[t:block_size+t+1]
  target = y[block_size+t]
  print(f"when input is {context} the target: {target}")

when input is tensor([42, 49, 48, 49, 39, 37,  0,  0,  7]) the target: 15
when input is tensor([49, 48, 49, 39, 37,  0,  0,  7, 15]) the target: 23
when input is tensor([48, 49, 39, 37,  0,  0,  7, 15, 23]) the target: 15
when input is tensor([49, 39, 37,  0,  0,  7, 15, 23, 15]) the target: 23
when input is tensor([39, 37,  0,  0,  7, 15, 23, 15, 23]) the target: 11
when input is tensor([37,  0,  0,  7, 15, 23, 15, 23, 11]) the target: 15
when input is tensor([ 0,  0,  7, 15, 23, 15, 23, 11, 15]) the target: 23
when input is tensor([ 0,  7, 15, 23, 15, 23, 11, 15, 23]) the target: 16


In [ ]:
torch.manual_seed(42)
batch_size = 4
block_size = 8

def get_batch(split):
  data = train_data if split == 'train' else val_data
  ix = torch.randint(len(data) - block_size, (batch_size,))
  x = torch.stack([data[i:i+block_size] for i in ix])
  y = torch.stack([data[i+1:i+block_size+1] for i in ix])
  return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('-----')

for b in range(batch_size):
  for t in range(block_size):
    context = xb[b, :t+1]
    target = yb[b,t]
    print(f"when input is {context.tolist()} the target: {target}")


inputs:
torch.Size([4, 8])
tensor([[ 12,   2,  30,  58,  75,  98, 100,   2],
        [ 58,  69,  98, 100,  98,  69,  70,  98],
        [ 58,  82,  66,  70,   2,  79,  62,   2],
        [ 70,  98, 100,  77,  98,  75,  12,   2]])
targets:
torch.Size([4, 8])
tensor([[  2,  30,  58,  75,  98, 100,   2,  68],
        [ 69,  98, 100,  98,  69,  70,  98, 100],
        [ 82,  66,  70,   2,  79,  62,   2,  71],
        [ 98, 100,  77,  98,  75,  12,   2,  97]])
-----
when input is [12] the target: 2
when input is [12, 2] the target: 30
when input is [12, 2, 30] the target: 58
when input is [12, 2, 30, 58] the target: 75
when input is [12, 2, 30, 58, 75] the target: 98
when input is [12, 2, 30, 58, 75, 98] the target: 100
when input is [12, 2, 30, 58, 75, 98, 100] the target: 2
when input is [12, 2, 30, 58, 75, 98, 100, 2] the target: 68
when input is [58] the target: 69
when input is [58, 69] the target: 98
when input is [58, 69, 98] the target: 100
when input is [58, 69, 98, 100] the target: 9

# Token Embeddibng:

In [ ]:
def _init_ (self):
  super()._init_()
  self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
  self.position_embedding_table = nn.Embedding(block_size, n_embd)
  self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
  sel.ln_f = nn.LayerNorm(n_embd) # final layer norm
  self.lm_head = nn.Linear(n_embd, vocab_size)

  self.apply(self._init_weights)

# Weighted Aggregation In Self Attention:

In [ ]:
torch.manual_seed(1337)
B,T,C = 4,8,2
x = torch.randn(B,T,C)
print(x.shape)
print(x[0])

torch.Size([4, 8, 2])
tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])


In [ ]:
torch.manual_seed(1337)
a = torch.tril(torch.ones(3,3))
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(0,10,(3,2)).float()
c = a @ b
print("a=")
print(a)
print("b=")
print(b)
print("c=")
print(c)

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
b=
tensor([[5., 7.],
        [2., 0.],
        [5., 3.]])
c=
tensor([[5.0000, 7.0000],
        [3.5000, 3.5000],
        [4.0000, 3.3333]])


In [ ]:
tril = torch.tril(torch.ones(T,T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf')) # mask all zero positions of tril of wei negative infinity
wei = F.softmax(wei, dim=-1)
xbow = wei @ x  # (T,T) @ (B,T,C)
print(wei)
print(x[0])
print(xbow[0])

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])
tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])
tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.09

## Self Attention

In [ ]:
# key: what we have
# query: what we looking for
# value = x değeri
# wei = query*key
# wei*x


In [ ]:
torch.manual_seed(1337)
B, T, C = 4,8,32
x = torch.randn(B,T,C)

#let's see a single head perform self-attention
head_size = 16
key  = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x) # b,t,16
q = query(x) #b,t,16
wei = q @ k.transpose(-2, -1) #b,t,16 @ b, 16, t = b, t


tril = torch.tril(torch.ones(T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

tril = torch.tril(torch.ones(T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)


v = value(x)
out = wei @ v

print(out.shape)
print(out[0])

torch.Size([4, 8, 16])
tensor([[-0.1571,  0.8801,  0.1615, -0.7824, -0.1429,  0.7468,  0.1007, -0.5239,
         -0.8873,  0.1907,  0.1762, -0.5943, -0.4812, -0.4860,  0.2862,  0.5710],
        [ 0.5006, -0.2466, -0.1615,  0.0830, -0.1311, -0.0755, -0.3178, -0.1964,
         -0.2260,  0.6137,  0.5997, -0.2172,  0.1563,  0.1683,  0.0044,  1.1238],
        [ 0.4476, -0.0802, -0.3119,  0.0955,  0.0699, -0.0908, -0.0592, -0.0645,
         -0.2826, -0.1032,  0.3499,  0.0248, -0.1936, -0.0363, -0.0802,  1.1567],
        [ 0.4068, -0.0845, -0.2871,  0.0318,  0.1930, -0.1398, -0.0345, -0.0953,
         -0.1938,  0.0910,  0.1266,  0.0054, -0.0561,  0.0650,  0.1895,  0.8319],
        [ 0.3765,  0.2044,  0.0621,  0.1823,  0.2566,  0.1835,  0.2193,  0.1309,
         -0.3135, -0.4260, -0.0834, -0.0718, -0.3925,  0.1358,  0.0538,  0.7487],
        [ 0.2218,  0.1336, -0.0230,  0.2807,  0.2616,  0.1399,  0.0904,  0.0177,
         -0.1801, -0.2140, -0.0273,  0.0754, -0.2312,  0.1449,  0.1918,  0.6403],

In [ ]:
n_emb = 256
dropout = 0.2

class Head(nn.Module):
  """ one head self attention """

  def __init__(self, head_size):
    super().__init__()
    key  = nn.Linear(C, head_size, bias=False)
    query = nn.Linear(C, head_size, bias=False)
    value = nn.Linear(C, head_size, bias=False)
    self.register_buffer('tril', torch.tril(torch.ones(block_size,block_size)))

    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    B,T,C = x.shape
    k = self.key(x)
    q = self.query(x)

    wei = q @ k.transpose(-2,1) * k.shape[-1]**-0.5
    wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
    wei = F.softmax(wei, dim=-1)
    wei = self.dropout(wei)

    v = self.value(x)
    out = wei @ v
    return out

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, head_size):
      super()._init_()
      self.heads = nn.ModuleList((Head(head_size) for _ in range(num_heads)))
      self.proj = nn.Linear(head_size * num_heads, n_embd)
      self.dropout = nn.Dropout(dropout)

    def forward(self, x):
      out = torch.cat([h(x) for h in self.heads], dim=-1)
      out = self.dropout(self.proj(out))
      return out

In [ ]:
# feed forward

class FeedForward(nn.Module):
  def __init__(self, n_embd):
    super()._init_()
    self.net = nn.Sequential(
        nn.Linear(n_embd, 4 * n_embd),
        nn.ReLU(),
        nn.Linear(4 * n_embd, n_embd),
        nn.Dropout(dropout),
    )
  def forward(self, x):
    return self.net(x)

In [ ]:
class Block(nn.Module):

  def _init_(self, n_embd, n_head):
    super()._init_()
    head_size = n_embd // n_head
    self.sa = MultiHeadAttention(n_head, head_size)
    self.ffwd

    self.ln1 = nn.LayerNorm(n_embd)
    self.ln2 = nn.LayerNorm(n_embd)

  def forward(self, x):
    x = x + self.sa(self.ln1(x)) #residual connection
    x = x + self.sa(self.ln2(x))
    return x


In [ ]:
#train

import os

import torch
import torch.nn as nn
from torch.nn import functional as F

#hyperparameters
batch_size = 64 # how many independent sequences will we process in parallel?
block_size = 256 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 500
eval_iters = 50
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
n_embd = 256
n_head = 6
n_layer = 6
dropout = 0.2


torch.manual_seed(1337)

input_path = "/content/drive/MyDrive/nutuk.txt"
with open(input_path) as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)

# create a mapping from characters to integers
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s] #encoder: take a string,output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) #decoder: take a list of integers, output a string


# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
  # generate a small batch of data of inputs x and targets y
  data = train_data if split == 'train' else val_data
  ix = torch.randint(len(data) - block_size, (batch_size,))
  x = torch.stack([data[i:i+block_size] for i in ix])
  y = torch.stack([data[i+1:i+block_size+1] for i in ix])
  x, y = x.to(device), y.to(device)
  return x, y

@torch.no_grad()
def estimate_loss():
  out = {}
  model.eval()
  for split in ['train', 'val']:
    losses = torch.zeros(eval_iters)
    for k in range(eval_iters):
      X, Y = get_batch(split)
      logits, loss = model(X, Y)
      losses[k] = loss.item()
    out[split] = losses.mean()
  model.train()
  return out

class Head(nn.Module):
  """ one head of self attention """

  def __init__(self, head_size):
    super().__init__()
    self.key = nn.Linear(n_embd, head_size, bias=False)
    self.query = nn.Linear(n_embd, head_size, bias=False)
    self.value = nn.Linear(n_embd, head_size, bias=False)
    self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    # input of size (batch, time-step, channels)
    #output of size (batch, time-step, channels)
    B,T,C = x.shape
    k = self.key(x)
    q = self.query(x)
    # compute attention scores ("affinities")
    wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5 # (B, T, hs) @ (B, hs, T) -> (B, T, T)
    wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
    wei = F.softmax(wei, dim=-1) # (B, T, T)
    wei = self.dropout(wei)
    # perform the weighted aggregation of the values
    v = self.value(x) # (B,T,hs)
    out = wei @ v # (B, T, T) @ (B, T, hs) -> (B, T, hs)
    return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """
    def __init__(self, num_heads, head_size):
      super().__init__()
      self.heads = nn.ModuleList((Head(head_size) for _ in range(num_heads)))
      self.proj = nn.Linear(head_size * num_heads, n_embd)
      self.dropout = nn.Dropout(dropout)

    def forward(self, x):
      out = torch.cat([h(x) for h in self.heads], dim=-1)
      out = self.dropout(self.proj(out))
      return out

# feed forward
class FeedForward(nn.Module):
    """ a simple linear layer followed by a non-linearity """
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )
    def forward(self, x):
      return self.net(x)

class Block(nn.Module):
  """ Transformer block: communication followed by computation """

  def __init__ (self, n_embd, n_head):
    super().__init__()
    head_size = n_embd // n_head
    self.sa = MultiHeadAttention(n_head, head_size)
    self.ffwd = FeedForward(n_embd)
    self.ln1 = nn.LayerNorm(n_embd)
    self.ln2 = nn.LayerNorm(n_embd)

  def forward(self, x):
    x = x + self.sa(self.ln1(x))
    x = x + self.ffwd(self.ln2(x))
    return x

class GPTLanguageModel(nn.Module):

  def __init__ (self):
    super().__init__()
    # each token directly reads off the logits for the next token from a lookup table
    self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
    self.position_embedding_table = nn.Embedding(block_size, n_embd)
    self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
    self.ln_f = nn.LayerNorm(n_embd) # final layer norm
    self.lm_head = nn.Linear(n_embd, vocab_size)

    # better init, not covered in the original GPT video, but important, will cover in followup video
    self.apply(self.__init__weights)

  def __init__weights(self, module):
    if isinstance(module, nn.Linear):
      torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
      if module.bias is not None:
        torch.nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Embedding):
      torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

  def forward(self, idx, target=None):

    B, T = idx.shape
    tok_emb = self.token_embedding_table(idx) # (B,T,C)
    pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
    x = tok_emb + pos_emb # (B,T,C)
    x = self.blocks(x) # (B,T,C)
    x = self.ln_f(x) # (B,T,C)
    logits = self.lm_head(x) # (B,T,vocab_size)

    if target is None:
        loss = None
    else:
        B, T, C = logits.shape
        logits = logits.view(B*T, C) # you should regularize the dimensions
        target = target.view(B*T)
        loss = F.cross_entropy(logits, target)

    return logits, loss

  def generate(self, idx, max_new_tokens):
         # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
          idx_cond = idx[:, -block_size:] # crop idx to the last block_size tokens
          logits, loss = self(idx_cond) # get the predictions
          logits = logits[:, -1, :] # focus only on the last time step # becomes (B, C)
          probs = F.softmax(logits, dim=-1) # apply softmax to get probabilities # (B, C)
          idx_next = torch.multinomial(probs, num_samples=1) # sample from the distribution  # (B, 1)
          idx = torch.cat((idx, idx_next), dim=1) # append sampled index to the running sequence # (B, T+1)
        return idx

model = GPTLanguageModel()
m = model.to(device)
#print the number of the parameters of the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

#create a pytorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

  #every once in a while evaluate the loss on train and val sets
  if iter % eval_interval == 0 or iter == max_iters - 1:
    losses = estimate_loss()
    print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

  #sample a batch of data
  xb, yb = get_batch('train')

  #evaluate the loss
  logits, loss = model(xb, yb)
  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()

#generate from the model
context = torch.zeros((1,1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))

# Save model to Drive
filename = os.path.basename(input_path).split('.')[0]
save_path = f'/content/drive/MyDrive/model_{filename}.pth'
torch.save(model.state_dict(), save_path)
print(f'Model saved to {save_path}')

4.828263 M parameters
step 0: train loss 4.7407, val loss 4.7398
step 500: train loss 1.8383, val loss 1.8999
step 1000: train loss 1.3273, val loss 1.4750
step 1500: train loss 1.2081, val loss 1.3743
step 2000: train loss 1.1284, val loss 1.3303
step 2500: train loss 1.0817, val loss 1.2941
step 3000: train loss 1.0451, val loss 1.2821
step 3500: train loss 1.0181, val loss 1.2693
step 4000: train loss 0.9838, val loss 1.2586
step 4500: train loss 0.9641, val loss 1.2472
step 4999: train loss 0.9404, val loss 1.2495


Maart Müfettiş, Tevfik Paşa, Tebi Ulağü, Bey, mukadderatını kesmektip okuydularından
Abdülke ve Rumeli binbaşısı , Atatürk'ün Bey ilisiste; Erzurum ve Erzurum ve
Türkislimini S49) kurtuluşunda ve Fırkalar Niyasetar'a kaldım.
Mali haberleşmesi saniye ve şanetimiş olan kendi tara­
rında, Kuvvei bir subaylar siyasi teklif merkezi gerekli evladır.
Bu prensipleri temsil edüşen meşrutiyetlerin ve böyle ikinci hürmeti mem­
leketine haklarınm, silah ve mevcudiyet ve tahasıslann

In [ ]:
class Block(nn.Module):

  def _init_ (self, n_embd, n_head):
    super()._init_()
    head_size = n_embd // n_head
    self.sa = MultiHeadAttention(n_head, head_size)
    self.ffwd = FeedForward(n_embd)

  def forward(self, x):
    x = x + self.sa(x)
    x = x + self.ffwd(x)
    return x